# probar modelos multirelacionales (normalizado entrenado con 2019-2020, probado en 2021)

In [2]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from types import SimpleNamespace
from sklearn.metrics import classification_report, precision_recall_fscore_support

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado\\"

TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_20192020_patience500_normalizado"
    r"\best_model.pt"
)

PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\results\predictor_models_multirrelacional"
    r"\predictor_rdim{rdim}\best_predictor_rdim{rdim}_normalizado.pt"
)

BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones (Clases)
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1,17)

# ============================================================
# 🔹 DEPENDENCIAS DE TUCKEr
# ============================================================

sys.path.append(r"C:\Users\56946\TuckER")
from load_data import Data

# ----- Predictor -----
class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x):
        return self.network(x)

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(predictor_path, map_location=device)
    if isinstance(state, dict) and any(k in state for k in ("model_state_dict","state_dict")):
        state = state.get("model_state_dict", state.get("state_dict"))
    model.load_state_dict(state)
    model.to(device).eval()
    return model

# ----- TuckER core -----
def pick_state_dict(ckpt):
    if "model_state_dict" in ckpt and isinstance(ckpt["model_state_dict"], dict):
        return ckpt["model_state_dict"]
    if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(tucker_path, device="cpu"):
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    E = state["E.weight"].to(device)
    R = state["R.weight"].to(device)
    W = state["W"].to(device)
    return E, R, W

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    entities, relations = d.entities, d.relations
    ent2idx = {e:i for i,e in enumerate(entities)}
    rel2idx = {r:i for i,r in enumerate(relations)}
    return SimpleNamespace(entities=entities, relations=relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands:
        raise ValueError(f"No hallé relación que contenga '{hint}'.")
    cands.sort(key=lambda x: (("_reverse" in x), x))
    chosen = cands[0]
    return rel2idx[chosen], chosen

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0]))
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else:
        raise ValueError(f"Layout W desconocido")

    v = e_hat @ M_r
    e_t = E[t_idx]
    logit = torch.dot(v, e_t)
    return float(logit)

def get_notes_vector(csv_path, alumno_id, cursos_primer):
    df = pd.read_csv(csv_path, sep=';')
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    if not subset.empty and subset['NOTA'].isna().all():
        subset['NOTA'] = 0.0
    if subset.empty:
        return torch.zeros((1, len(cursos_primer)), dtype=torch.float32)
    pivot = (subset
             .pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
             .reindex(columns=cursos_primer)
             .fillna(0.0))
    vec = pivot.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1, -1)

# ============================================================
# 🔹 CARGA Y PREPARACIÓN DE DATOS
# ============================================================

df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')
for df in (df_20211, df_20212):
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()
print(f"🔹 {len(alumnos_validos)} alumnos válidos 20211.")

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(nota):
    if pd.isna(nota) or nota < 4.0:
        return "reprueba"
    elif nota < 5.0:
        return "aprueba_4_5"
    elif nota < 6.0:
        return "aprueba_5_6"
    else:
        return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval["NOTA"].apply(categoria_relacion)
print("📊 Distribución de relaciones reales (20212):")
print(df_eval["RELACION_REAL"].value_counts())

# ============================================================
# 🔹 FUNCIÓN DE RANKING MODIFICADA
# ============================================================

def rank_relations_for_row(e_hat, course_idx, rel2idx_map, R, W, E, true_rel_hint):
    scores = {}
    for rel_hint, r_idx in rel2idx_map.items():
        s = tucker_score_single(e_hat, R, W, E, r_idx, course_idx)
        scores[rel_hint] = s

    # Ranking descendente
    orden = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    rels_orden = [r for r, _ in orden]

    # Identificamos la predicción TOP 1
    top1_prediction = rels_orden[0]

    if true_rel_hint not in rels_orden:
        return np.nan, 0.0, 0.0, 0.0, np.nan, None 

    pos = rels_orden.index(true_rel_hint) + 1
    hit1 = 1.0 if pos <= 1 else 0.0
    hit2 = 1.0 if pos <= 2 else 0.0
    hit3 = 1.0 if pos <= 3 else 0.0
    
    # Retornamos también la predicción
    return float(pos), hit1, hit2, hit3, float(scores[true_rel_hint]), top1_prediction

def imprime_metricas_relrank(df, titulo):
    if df.empty or df["RANK"].dropna().empty:
        return None
    ranks = df["RANK"].dropna().astype(float).to_numpy()
    m = {
        "hits@1": float(df["HIT1"].mean()),
        "hits@2": float(df["HIT2"].mean()),
        "hits@3": float(df["HIT3"].mean()),
        "mr": float(np.mean(ranks)),
        "mrr": float(np.mean(1.0 / ranks)),
        "n": int(len(ranks)),
    }
    return m

# ============================================================
# 🔹 LOOP PRINCIPAL POR rdim
# ============================================================

resumen_metricas = []
out_detalles = []

for rdim in RDIMS:
    print("\n\n==============================")
    print(f"  🔹 Evaluando rdim={rdim}")
    print("==============================")

    tucker_path    = TUCKER_DIR_FMT.format(rdim=rdim)
    predictor_path = PRED_DIR_FMT.format(rdim=rdim)

    if not os.path.exists(tucker_path) or not os.path.exists(predictor_path):
        print(f"⚠️  Faltan archivos para rdim={rdim}")
        continue

    d = build_vocab(DATA_DIR, reverse=True)
    E, R, W = load_tucker_weights(tucker_path, device="cpu")
    d1 = E.shape[1]

    rel2idx_map = {}
    for rel_hint in RELACIONES:
        idx, real_name = find_relation(d.relations, d.relation_idxs, hint=rel_hint)
        rel2idx_map[rel_hint] = idx

    predictor = load_predictor(predictor_path, input_size=len(CURSOS_PRIMER), out_dim=d1, device="cpu")

    ehat_by_alumno = {}
    for aid in df_eval["ID"].unique():
        x = get_notes_vector(CSV_20211, aid, CURSOS_PRIMER)
        with torch.no_grad():
            e_hat = predictor(x).squeeze(0)
        ehat_by_alumno[aid] = e_hat

    # Lista para almacenar filas con predicción
    filas = []
    
    for _, row in df_eval.iterrows():
        aid      = row["ID"]
        curso    = row["CURSO"]
        true_rel = row["RELACION_REAL"]

        if curso not in d.entity_idxs: continue
        t_idx = d.entity_idxs[curso]
        e_hat = ehat_by_alumno[aid]

        rank, h1, h2, h3, score_true, pred_rel = rank_relations_for_row(
            e_hat=e_hat, course_idx=t_idx, rel2idx_map=rel2idx_map,
            R=R, W=W, E=E, true_rel_hint=true_rel
        )
        
        filas.append({
            "ID": aid,
            "CURSO": curso,
            "RELACION_REAL": true_rel,
            "RELACION_PRED": pred_rel, # La que ganó (Top 1)
            "RANK": rank,
            "HIT1": h1, "HIT2": h2, "HIT3": h3,
            "SCORE_TRUE": score_true,
            "rdim": rdim
        })

    if not filas:
        continue

    df_det = pd.DataFrame(filas)

    # 1. Métricas de Ranking (Hits, MRR) - Como antes
    met_g = imprime_metricas_relrank(df_det, "GLOBAL")
    if met_g:
        met_g.update({"rdim": rdim, "relacion": "GLOBAL_RANKING"})
        resumen_metricas.append(met_g)

    # 2. Métricas de Clasificación (Precision, Recall)
    # Calculamos P/R/F1 para cada clase
    y_true = df_det["RELACION_REAL"]
    y_pred = df_det["RELACION_PRED"]
    
    # Calculamos métricas por etiqueta
    precision, recall, fscore, support = precision_recall_fscore_support(
        y_true, y_pred, labels=RELACIONES, zero_division=0
    )
    
    print("\n--- 📊 Métricas de Clasificación por Relación ---")
    print(f"{'Relación':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
    print("-" * 60)
    
    for i, rel in enumerate(RELACIONES):
        p, r, f, s = precision[i], recall[i], fscore[i], support[i]
        print(f"{rel:<15} {p:.4f}     {r:.4f}     {f:.4f}     {s}")
        
        # Guardamos en el resumen general
        met_class = {
            "rdim": rdim,
            "relacion": rel,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": s,
            # Hits también son útiles por relación, los traemos del ranking
            "hits@1": df_det[df_det["RELACION_REAL"]==rel]["HIT1"].mean(),
            "mr": df_det[df_det["RELACION_REAL"]==rel]["RANK"].mean()
        }
        resumen_metricas.append(met_class)

    # Guardar detalle
    out_det = os.path.join(os.getcwd(), f"metricas_detalle_completo_rdim{rdim}.csv")
    df_det.to_csv(out_det, index=False, encoding="utf-8")
    print(f"\n💾 Detalle guardado en: {out_det}")

# ============================================================
# 🔹 GUARDAR RESUMEN FINAL
# ============================================================
if resumen_metricas:
    df_resumen = pd.DataFrame(resumen_metricas)
    # Reordenar columnas para legibilidad
    cols = ["rdim", "relacion", "precision", "recall", "f1", "hits@1", "mr", "support"]
    # Aseguramos que existan las columnas (algunas filas son de ranking global y no tienen p/r)
    for c in cols:
        if c not in df_resumen.columns: df_resumen[c] = np.nan
            
    out_resumen = os.path.join(os.getcwd(), "metricas_resumen_clasificacion_ranking.csv")
    df_resumen[cols].to_csv(out_resumen, index=False, encoding="utf-8")
    print(f"\n✅ Resumen consolidado guardado en: {out_resumen}")
else:
    print("\n⚠️ No se generaron métricas.")

🔹 830 alumnos válidos 20211.
📊 Distribución de relaciones reales (20212):
aprueba_5_6    1345
aprueba_6_7     870
aprueba_4_5     575
reprueba        233
Name: RELACION_REAL, dtype: int64


  🔹 Evaluando rdim=1

--- 📊 Métricas de Clasificación por Relación ---
Relación        Precision  Recall     F1-Score   Support   
------------------------------------------------------------
reprueba        0.0000     0.0000     0.0000     233
aprueba_4_5     0.0000     0.0000     0.0000     575
aprueba_5_6     0.4449     1.0000     0.6158     1345
aprueba_6_7     0.0000     0.0000     0.0000     870

💾 Detalle guardado en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\metricas_detalle_completo_rdim1.csv


  🔹 Evaluando rdim=2

--- 📊 Métricas de Clasificación por Relación ---
Relación        Precision  Recall     F1-Score   Support   
------------------------------------------------------------
reprueba        0.1041     0.3948     0.1647     233
aprueba_4_5     0.1429     


--- 📊 Métricas de Clasificación por Relación ---
Relación        Precision  Recall     F1-Score   Support   
------------------------------------------------------------
reprueba        0.0000     0.0000     0.0000     233
aprueba_4_5     0.2771     0.3426     0.3064     575
aprueba_5_6     0.3143     0.1799     0.2288     1345
aprueba_6_7     0.2536     0.4494     0.3242     870

💾 Detalle guardado en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\metricas_detalle_completo_rdim15.csv


  🔹 Evaluando rdim=16

--- 📊 Métricas de Clasificación por Relación ---
Relación        Precision  Recall     F1-Score   Support   
------------------------------------------------------------
reprueba        0.0000     0.0000     0.0000     233
aprueba_4_5     0.4397     0.0887     0.1476     575
aprueba_5_6     0.5058     0.5539     0.5287     1345
aprueba_6_7     0.4066     0.6701     0.5061     870

💾 Detalle guardado en: C:\Users\56946\TuckER\notebooks\Experimento_warm_sta

# probar modelos multirelacionales (normalizado entrenado con 2019-2020, probado en 2021) colapsando relacionas aprueba

In [1]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado\\"

TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_20192020_patience500_normalizado"
    r"\best_model.pt"
)

PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\results\predictor_models_multirrelacional"
    r"\predictor_rdim{rdim}\best_predictor_rdim{rdim}_normalizado.pt"
)

BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones originales del modelo
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1,17)

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

sys.path.append(r"C:\Users\56946\TuckER")
from load_data import Data

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

# Función robusta para extraer pesos
def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

# ⚠️ CORREGIDO AQUÍ: Usamos pick_state_dict correctamente
def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(predictor_path, map_location=device)
    
    # Extraer state_dict correctamente
    state = pick_state_dict(state)
    
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    entities, relations = d.entities, d.relations
    ent2idx = {e:i for i,e in enumerate(entities)}
    rel2idx = {r:i for i,r in enumerate(relations)}
    return SimpleNamespace(entities=entities, relations=relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: raise ValueError(f"No hallé relación que contenga '{hint}'.")
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0]))
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: raise ValueError("Layout W desconocido")
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

def get_notes_vector(csv_path, alumno_id, cursos_primer):
    df = pd.read_csv(csv_path, sep=';')
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    if not subset.empty and subset['NOTA'].isna().all(): subset['NOTA'] = 0.0
    if subset.empty: return torch.zeros((1, len(cursos_primer)), dtype=torch.float32)
    pivot = (subset.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
             .reindex(columns=cursos_primer).fillna(0.0))
    vec = pivot.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1, -1)

# ============================================================
# 🔹 LÓGICA DE AGREGACIÓN BINARIA
# ============================================================

def es_reprobado(relacion_str):
    """Retorna True si la relación es 'reprueba', False si es cualquier 'aprueba...'"""
    return "reprueba" in relacion_str.lower()

# ============================================================
# 🔹 CARGA DATOS
# ============================================================

print("Cargando Dataframes...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')
for df in (df_20211, df_20212):
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(nota):
    if pd.isna(nota) or nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval["NOTA"].apply(categoria_relacion)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 EVALUACIÓN
# ============================================================

def main():
    print("\n--- INICIANDO EVALUACIÓN BINARIZADA (Aprueba vs Reprueba) ---")
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
        predictor_path = PRED_DIR_FMT.format(rdim=rdim)

        if not os.path.exists(tucker_path) or not os.path.exists(predictor_path):
            print(f"⚠️ Archivos no encontrados para rdim={rdim}")
            continue

        d = build_vocab(DATA_DIR, reverse=True)
        E, R, W = load_tucker_weights(tucker_path, device="cpu")
        d1 = E.shape[1]

        rel2idx_map = {}
        for rel_hint in RELACIONES:
            rel2idx_map[rel_hint] = find_relation(d.relations, d.relation_idxs, hint=rel_hint)

        predictor = load_predictor(predictor_path, input_size=len(CURSOS_PRIMER), out_dim=d1, device="cpu")

        # Cache embeddings
        ehat_by_alumno = {}
        for aid in df_eval["ID"].unique():
            x = get_notes_vector(CSV_20211, aid, CURSOS_PRIMER)
            with torch.no_grad(): ehat_by_alumno[aid] = predictor(x).squeeze(0)

        # Listas para métricas binarias
        y_true_bin = [] # 1 = Reprueba, 0 = Aprueba
        y_pred_bin = [] 

        for _, row in df_eval.iterrows():
            aid = row["ID"]
            curso = row["CURSO"]
            true_rel_str = row["RELACION_REAL"]

            if curso not in d.entity_idxs: continue
            
            # 1. Obtenemos la predicción MULTIRRELACIONAL (la que gana entre las 4)
            t_idx = d.entity_idxs[curso]
            e_hat = ehat_by_alumno[aid]
            
            scores = {}
            for r_name, r_idx in rel2idx_map.items():
                scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
            
            # La relación ganadora (string)
            pred_rel_str = max(scores, key=scores.get)
            
            # 2. CONVERSIÓN A BINARIO (COLLAPSE)
            # Definimos: Reprueba = 1 (Clase Positiva/Interés), Aprueba = 0
            
            # Realidad
            is_real_repro = 1 if es_reprobado(true_rel_str) else 0
            
            # Predicción (Si predijo 'reprueba' es 1, si predijo cualquier 'aprueba...' es 0)
            is_pred_repro = 1 if es_reprobado(pred_rel_str) else 0
            
            y_true_bin.append(is_real_repro)
            y_pred_bin.append(is_pred_repro)

        # 3. CÁLCULO DE MÉTRICAS BINARIAS
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
        
        # En nuestra lógica (1=Reprueba):
        # TP: Real Reprueba, Predijo Reprueba (Correcto)
        # FN: Real Reprueba, Predijo Aprueba (Peligroso)
        # FP: Real Aprueba, Predijo Reprueba (Falsa Alarma)
        # TN: Real Aprueba, Predijo Aprueba (Correcto)
        
        total_reales_reprobados = tp + fn
        total_predichos_reprobados = tp + fp
        
        recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0.0
        precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0.0
        accuracy = (tp + tn) / len(y_true_bin)
        
        print(f"📊 Resultados BINARIZADOS (Agregando los 3 'Aprueba' vs 'Reprueba')")
        print("-" * 60)
        print(f"Accuracy Global         : {accuracy:.4f}")
        print(f"Total REALES Reprobados : {total_reales_reprobados}")
        print("-" * 60)
        print(f"✅ RECALL (Sensibilidad) : {recall:.4f}")
        print(f"   (Detectamos {tp} de {total_reales_reprobados} reprobados)")
        print("-" * 60)
        print(f"🎯 PRECISION             : {precision:.4f}")
        print(f"   (De las {total_predichos_reprobados} alertas, {tp} eran reales)")
        print("-" * 60)
        print(f"Matriz:")
        print(f"   [TP={tp}] (Detectados)      [FN={fn}] (Perdidos/Peligro)")
        print(f"   [FP={fp}] (Falsa Alarma)    [TN={tn}] (Correctos Aprobados)")

if __name__ == "__main__":
    main()

Cargando Dataframes...
Evaluaciones totales: 3023

--- INICIANDO EVALUACIÓN BINARIZADA (Aprueba vs Reprueba) ---

============================== Evaluando rdim=1 ==============================
📊 Resultados BINARIZADOS (Agregando los 3 'Aprueba' vs 'Reprueba')
------------------------------------------------------------
Accuracy Global         : 0.9229
Total REALES Reprobados : 233
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.0000
   (Detectamos 0 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION             : 0.0000
   (De las 0 alertas, 0 eran reales)
------------------------------------------------------------
Matriz:
   [TP=0] (Detectados)      [FN=233] (Perdidos/Peligro)
   [FP=0] (Falsa Alarma)    [TN=2790] (Correctos Aprobados)

============================== Evaluando rdim=2 ==============================
📊 Resultados BINARIZADOS (Agregando los 3 'Aprueba' vs 'Reprueba')
-----------------------

📊 Resultados BINARIZADOS (Agregando los 3 'Aprueba' vs 'Reprueba')
------------------------------------------------------------
Accuracy Global         : 0.5240
Total REALES Reprobados : 233
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.4034
   (Detectamos 94 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION             : 0.0674
   (De las 1394 alertas, 94 eran reales)
------------------------------------------------------------
Matriz:
   [TP=94] (Detectados)      [FN=139] (Perdidos/Peligro)
   [FP=1300] (Falsa Alarma)    [TN=1490] (Correctos Aprobados)

============================== Evaluando rdim=14 ==============================
📊 Resultados BINARIZADOS (Agregando los 3 'Aprueba' vs 'Reprueba')
------------------------------------------------------------
Accuracy Global         : 0.9229
Total REALES Reprobados : 233
------------------------------------------------------------
✅ RECALL (Sensibilida

# Probando modelo data augmentation-multirelacional

In [3]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

# Dataset Multirrelacional (El mismo usado para entrenar el TuckER 2019-2020)
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"

# Ruta al TuckER Balanceado (Paciencia 400 - 2019_2020)
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado"
    r"\best_model.pt"
)

# ⚠️ CORRECCIÓN: Ruta EXACTA donde se guardaron tus redes neuronales (con _20192020)
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados_20192020"
    r"\best_predictor_dim4_rdim{rdim}_2019_balanceado.pt"
)

# Datos de evaluación (2021)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones del modelo multirrelacional
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17) 

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        # ⚠️ Input size dinámico (será 4 para S1->S2)
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(predictor_path, map_location=device)
    state = pick_state_dict(state)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

def get_notes_vector(csv_path, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, len(cursos_primer)), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    if subset.empty: return torch.zeros((1, len(cursos_primer)), dtype=torch.float32)
    
    pivot = (subset.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
             .reindex(columns=cursos_primer).fillna(0.0))
    vec = pivot.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1, -1)

# ============================================================
# 🔹 AGREGACIÓN BINARIA
# ============================================================
def es_reprobado(relacion_str):
    return "reprueba" in relacion_str.lower()

# ============================================================
# 🔹 CARGA DATOS
# ============================================================
print("Cargando Dataframes 2021...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

for df in (df_20211, df_20212):
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(row):
    nota = pd.to_numeric(row['NOTA'], errors='coerce')
    estado = str(row['ESTADO_CURSO'])
    
    if "Reprobado" in estado: return "reprueba"
    if pd.isna(nota): return "reprueba" 
    
    if nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval.apply(categoria_relacion, axis=1)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- INICIANDO EVALUACIÓN BINARIZADA (Modelo 2019-2020 Balanceado S1->S2) ---")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
        predictor_path = PRED_DIR_FMT.format(rdim=rdim)

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        E, R, W = load_tucker_weights(tucker_path, DEVICE)
        d1 = E.shape[1]

        # Mapear relaciones
        rel2idx_map = {}
        for rel_hint in RELACIONES:
            idx = find_relation(vocab.relations, vocab.relation_idxs, hint=rel_hint)
            if idx is not None:
                rel2idx_map[rel_hint] = idx

        # ⚠️ INPUT SIZE = 4 (Porque usamos solo el 1er semestre para predecir)
        predictor = load_predictor(predictor_path, input_size=4, out_dim=d1, device="cpu")

        # Precalcular embeddings (Input Dim 4)
        ehat_cache = {}
        for aid in df_eval["ID"].unique():
            x = get_notes_vector(CSV_20211, aid, CURSOS_PRIMER)
            with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

        y_true_bin = []
        y_pred_bin = []

        for _, row in df_eval.iterrows():
            aid = row["ID"]
            curso = row["CURSO"]
            true_rel = row["RELACION_REAL"]

            if curso not in vocab.entity_idxs: continue
            
            t_idx = vocab.entity_idxs[curso]
            e_hat = ehat_cache[aid]
            
            # Calcular score para las 4 relaciones
            scores = {}
            for r_name, r_idx in rel2idx_map.items():
                scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
            
            # Ganadora
            pred_rel = max(scores, key=scores.get)
            
            # Binarizar
            real_bin = 1 if es_reprobado(true_rel) else 0
            pred_bin = 1 if es_reprobado(pred_rel) else 0
            
            y_true_bin.append(real_bin)
            y_pred_bin.append(pred_bin)

        # Métricas
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
        
        total_reales_reprobados = tp + fn
        total_predichos_reprobados = tp + fp
        
        recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
        precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
        accuracy = (tp + tn) / len(y_true_bin)
        
        print(f"📊 Resultados Multirrelacional -> Binario (rdim={rdim})")
        print("-" * 60)
        print(f"Accuracy Global         : {accuracy:.4f}")
        print(f"Total REALES Reprobados : {total_reales_reprobados}")
        print("-" * 60)
        print(f"✅ RECALL (Sensibilidad) : {recall:.4f}")
        print(f"   (Detectamos {tp} de {total_reales_reprobados} reprobados)")
        print("-" * 60)
        print(f"🎯 PRECISION             : {precision:.4f}")
        print(f"   (De las {total_predichos_reprobados} alertas, {tp} eran reales)")
        print("-" * 60)
        print(f"Matriz:")
        print(f"   [TP={tp}] (Detectados)      [FN={fn}] (Perdidos/Peligro)")
        print(f"   [FP={fp}] (Falsa Alarma)    [TN={tn}] (Correctos Aprobados)")

if __name__ == "__main__":
    main()

Cargando Dataframes 2021...
Evaluaciones totales: 3023

--- INICIANDO EVALUACIÓN BINARIZADA (Modelo 2019-2020 Balanceado S1->S2) ---

============================== Evaluando rdim=1 ==============================
📊 Resultados Multirrelacional -> Binario (rdim=1)
------------------------------------------------------------
Accuracy Global         : 0.0771
Total REALES Reprobados : 233
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 1.0000
   (Detectamos 233 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION             : 0.0771
   (De las 3023 alertas, 233 eran reales)
------------------------------------------------------------
Matriz:
   [TP=233] (Detectados)      [FN=0] (Perdidos/Peligro)
   [FP=2790] (Falsa Alarma)    [TN=0] (Correctos Aprobados)

============================== Evaluando rdim=2 ==============================
📊 Resultados Multirrelacional -> Binario (rdim=2)
------------------------------

📊 Resultados Multirrelacional -> Binario (rdim=13)
------------------------------------------------------------
Accuracy Global         : 0.2961
Total REALES Reprobados : 233
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.5408
   (Detectamos 126 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION             : 0.0587
   (De las 2147 alertas, 126 eran reales)
------------------------------------------------------------
Matriz:
   [TP=126] (Detectados)      [FN=107] (Perdidos/Peligro)
   [FP=2021] (Falsa Alarma)    [TN=769] (Correctos Aprobados)

============================== Evaluando rdim=14 ==============================
📊 Resultados Multirrelacional -> Binario (rdim=14)
------------------------------------------------------------
Accuracy Global         : 0.0771
Total REALES Reprobados : 233
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 1.0000
   (Detectamos 233

# Colapsando reprueba y aprueba con [4,5)

In [4]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

# Dataset Multirrelacional
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"

# Ruta al TuckER Balanceado (Paciencia 400 - 2019_2020)
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado"
    r"\best_model.pt"
)

# Ruta a los Predictores Neuronales
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados_20192020"
    r"\best_predictor_dim4_rdim{rdim}_2019_balanceado.pt"
)

# Datos de evaluación (2021)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones del modelo
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17) 

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(predictor_path, map_location=device)
    state = pick_state_dict(state)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

def get_notes_vector(csv_path, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, len(cursos_primer)), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    if subset.empty: return torch.zeros((1, len(cursos_primer)), dtype=torch.float32)
    
    pivot = (subset.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
             .reindex(columns=cursos_primer).fillna(0.0))
    vec = pivot.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1, -1)

# ============================================================
# 🔹 LÓGICA DE COLAPSO (RIESGO AMPLIADO)
# ============================================================
def es_riesgo_ampliado(relacion_str):
    """
    Retorna True si la relación indica RIESGO.
    RIESGO = 'reprueba' O 'aprueba_4_5' (Aprobar raspando)
    NO RIESGO = 'aprueba_5_6' O 'aprueba_6_7'
    """
    rel = relacion_str.lower()
    return "reprueba" in rel or "aprueba_4_5" in rel

# ============================================================
# 🔹 CARGA DATOS
# ============================================================
print("Cargando Dataframes 2021...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

for df in (df_20211, df_20212):
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(row):
    nota = pd.to_numeric(row['NOTA'], errors='coerce')
    estado = str(row['ESTADO_CURSO'])
    
    if "Reprobado" in estado: return "reprueba"
    if pd.isna(nota): return "reprueba" 
    
    if nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval.apply(categoria_relacion, axis=1)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- INICIANDO EVALUACIÓN RIESGO AMPLIADO (Reprueba + [4,5)) ---")
    print("Objetivo: Ver si colapsar las clases bajas mejora el Recall/Precision")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
        predictor_path = PRED_DIR_FMT.format(rdim=rdim)

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        E, R, W = load_tucker_weights(tucker_path, DEVICE)
        d1 = E.shape[1]

        # Mapear relaciones
        rel2idx_map = {}
        for rel_hint in RELACIONES:
            idx = find_relation(vocab.relations, vocab.relation_idxs, hint=rel_hint)
            if idx is not None:
                rel2idx_map[rel_hint] = idx

        # Predictor
        predictor = load_predictor(predictor_path, input_size=4, out_dim=d1, device="cpu")

        # Precalcular embeddings
        ehat_cache = {}
        for aid in df_eval["ID"].unique():
            x = get_notes_vector(CSV_20211, aid, CURSOS_PRIMER)
            with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

        y_true_collapsed = []
        y_pred_collapsed = []

        for _, row in df_eval.iterrows():
            aid = row["ID"]
            curso = row["CURSO"]
            true_rel = row["RELACION_REAL"]

            if curso not in vocab.entity_idxs: continue
            
            t_idx = vocab.entity_idxs[curso]
            e_hat = ehat_cache[aid]
            
            # Calcular score para las 4 relaciones
            scores = {}
            for r_name, r_idx in rel2idx_map.items():
                scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
            
            # Ganadora (multiclase)
            pred_rel = max(scores, key=scores.get)
            
            # ⚠️ COLAPSO DE CLASES ⚠️
            # 1 = RIESGO (Reprueba O Aprueba 4.0-5.0)
            # 0 = OK (Aprueba > 5.0)
            real_risk = 1 if es_riesgo_ampliado(true_rel) else 0
            pred_risk = 1 if es_riesgo_ampliado(pred_rel) else 0
            
            y_true_collapsed.append(real_risk)
            y_pred_collapsed.append(pred_risk)

        # Métricas
        tn, fp, fn, tp = confusion_matrix(y_true_collapsed, y_pred_collapsed).ravel()
        
        total_reales_riesgo = tp + fn
        total_predichos_riesgo = tp + fp
        
        recall = tp / total_reales_riesgo if total_reales_riesgo > 0 else 0
        precision = tp / total_predichos_riesgo if total_predichos_riesgo > 0 else 0
        accuracy = (tp + tn) / len(y_true_collapsed)
        
        print(f"📊 Resultados RIESGO AMPLIADO (rdim={rdim})")
        print("-" * 60)
        print(f"Accuracy Global          : {accuracy:.4f}")
        print(f"Total REALES en Riesgo   : {total_reales_riesgo} (Repro + Nota 4-5)")
        print("-" * 60)
        print(f"✅ RECALL (Sensibilidad) : {recall:.4f}")
        print(f"   (Detectamos {tp} de {total_reales_riesgo} casos de riesgo ampliado)")
        print("-" * 60)
        print(f"🎯 PRECISION             : {precision:.4f}")
        print(f"   (De las {total_predichos_riesgo} alertas, {tp} eran reales)")
        print("-" * 60)
        print(f"Matriz:")
        print(f"   [TP={tp}] (Riesgo Detectado)    [FN={fn}] (Riesgo No Visto)")
        print(f"   [FP={fp}] (Falsa Alarma)        [TN={tn}] (Correctos OK)")

if __name__ == "__main__":
    main()

Cargando Dataframes 2021...
Evaluaciones totales: 3023

--- INICIANDO EVALUACIÓN RIESGO AMPLIADO (Reprueba + [4,5)) ---
Objetivo: Ver si colapsar las clases bajas mejora el Recall/Precision

============================== Evaluando rdim=1 ==============================
📊 Resultados RIESGO AMPLIADO (rdim=1)
------------------------------------------------------------
Accuracy Global          : 0.2673
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 1.0000
   (Detectamos 808 de 808 casos de riesgo ampliado)
------------------------------------------------------------
🎯 PRECISION             : 0.2673
   (De las 3023 alertas, 808 eran reales)
------------------------------------------------------------
Matriz:
   [TP=808] (Riesgo Detectado)    [FN=0] (Riesgo No Visto)
   [FP=2215] (Falsa Alarma)        [TN=0] (Correctos OK)

============================== Evaluando rdim=2 ==============================


📊 Resultados RIESGO AMPLIADO (rdim=12)
------------------------------------------------------------
Accuracy Global          : 0.2673
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 1.0000
   (Detectamos 808 de 808 casos de riesgo ampliado)
------------------------------------------------------------
🎯 PRECISION             : 0.2673
   (De las 3023 alertas, 808 eran reales)
------------------------------------------------------------
Matriz:
   [TP=808] (Riesgo Detectado)    [FN=0] (Riesgo No Visto)
   [FP=2215] (Falsa Alarma)        [TN=0] (Correctos OK)

============================== Evaluando rdim=13 ==============================
📊 Resultados RIESGO AMPLIADO (rdim=13)
------------------------------------------------------------
Accuracy Global          : 0.2673
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad)